# SQL SubQuery
## How to load the dataset(s) to SQL?
Using Python, you can load the large datasets to MySQL database very easily. For that follow the below steps.

- First create a database in your local machine server.

```sql
CREATE DATABASE <database_name>
```

- Next, use Python to load the database

```python
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv("file/path/to/the/database.csv")

engine = create_engine("mysql+pymysql://<db_username>:<db_password>@<hostname>/<database_name>")
df.to_sql("<table_name>", con=engine)
```

## Problems 1-6

For problems 1 to 6, use the Olympic dataset. You can get that from [here](https://drive.google.com/file/d/1EGIRBkbQGByJPvCqDtxtTnXv93oGunFp/view?usp=share_link).

**Column description:**
1. ID -> ID of every records to our dataset. It has integer datatype.
2. Name -> Name of the athletes.
3. Sex -> Gender of the athletes.
4. Height -> Height of the athletes
5. Weight -> Weight of the athletes
6. NOC -> In which country, the athletes belong to. This is actually the country code.
7. Year -> In which year, the athlete has participated
8. Sport -> What is the sport name in which the athlete participated.
9. Event -> Event name of the sport
10. Medal -> Which medal the athlege got. If the athlete did not get any medal then this cell is blank.
11. country -> The name of the country.

### Problem 1

Display the names of athletes who won a gold medal in the 2008 Olympics and whose height is greater than the average height of all athletes in the 2008 Olympics.


### Problem 2

Display the names of athletes who won a medal in the sport of basketball in the 2016 Olympics and whose weight is less than the average weight of all athletes who won a medal in the 2016 Olympics.



### Problem 3

Display the names of all athletes who have won a medal in the sport of swimming in both the 2008 and 2016 Olympics.



### Problem 4

Display the names of all countries that have won more than 50 medals in a single year.



### Problem 5

Display the names of all athletes who have won medals in more than one sport in the same year.



### Problem 6

What is the average weight difference between male and female athletes in the Olympics who have won a medal in the same event?

In [ ]:
SELECT COUNT(*)
FROM athlete_events;

-- change type of age col
ALTER TABLE athlete_events
ALTER COLUMN age TYPE INTEGER
USING NULLIF(age, 'NA')::integer;

-- change type of height col
ALTER TABLE athlete_events
ALTER COLUMN height TYPE INTEGER
USING NULLIF(height, 'NA')::integer;

-- change type of weight col
ALTER TABLE athlete_events
ALTER COLUMN weight TYPE DECIMAL
USING NULLIF(weight, 'NA')::decimal;

In [ ]:
-- Problems 1-6 from Olympic Dataset
-- problem 1
SELECT *
FROM athlete_events
WHERE medal = 'Gold'
  AND year = '2008'
  AND height > (SELECT AVG(height)
                FROM athlete_events
                WHERE year = '2008');

-- problem 2
SELECT *
FROM athlete_events
WHERE sport = 'Basketball'
  AND medal != 'NA'
  AND year = '2016'
  AND weight < (SELECT AVG(weight)
                FROM athlete_events
                WHERE year = '2016' AND medal != 'NA');

-- problem 3
SELECT *
FROM athlete_events
WHERE medal != 'NA'
  AND sport = 'Swimming'
  AND year IN ('2008', '2016');

-- problem 4
SELECT noc,
       year,
       COUNT(*) AS medal_count
FROM athlete_events
WHERE medal != 'NA'
GROUP BY noc, year
HAVING COUNT(*) > 50;

-- problem 5
SELECT name,
       sport,
       year,
       COUNT(*)
FROM athlete_events
WHERE medal != 'NA'
GROUP BY name, sport, year;

-- problem 6
SELECT AVG(A.weight - B.weight)
FROM (
    SELECT *
    FROM athlete_events
    WHERE medal IS NOT NULL AND sex = 'F'
     ) A
JOIN (
    SELECT *
    FROM athlete_events
    WHERE medal IS NOT NULL AND sex = 'M'
     ) B
ON A.event = B.event;

-- the most optimal query for the given problem
SELECT AVG(A.weight - B.weight)
FROM athlete_events A
JOIN athlete_events B ON A.event = B.event
WHERE A.medal IS NOT NULL AND A.sex = 'F'
  AND B.medal IS NOT NULL AND B.sex = 'M';

## Problem 7 - 10

Use the health insurance dataset. You can get the dataset as well as the description of the dataset [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### Problem 7

How many patients have claimed more than the average claim amount for patients who are smokers, have at least one child, and belong to the southeast region?


### Problem 8

How many patients have claimed more than the average claim amount for patients who are not smokers and have a BMI greater than the average BMI for patients who have at least one child?



### Problem 9

How many patients have claimed more than the average claim amount for patients who have a BMI greater than the average BMI for patients who are diabetic, have at least one child, and are from the southwest region?


### Problem 10:

What is the difference in the average claim amount between patients who are smokers and patients who are non-smokers, and have the same BMI and number of children?

In [ ]:
-- Problems 7-10 from Insurance Dataset
-- problem 7
SELECT COUNT(*)
FROM insurance_data
WHERE claim > (SELECT AVG(claim)
               FROM insurance_data
               WHERE smoker = 'Yes'
                 AND children >= 1
                 AND region = 'southeast');

-- problem 8
SELECT COUNT(*)
FROM insurance_data
WHERE claim > (SELECT AVG(claim)
               FROM insurance_data
               WHERE smoker = 'No'
                 AND bmi > (SELECT AVG(bmi)
                            FROM insurance_data
                            WHERE children >= 1));

-- problem 9
SELECT COUNT(*)
FROM insurance_data
WHERE claim > (SELECT AVG(claim)
               FROM insurance_data
               WHERE bmi > (SELECT AVG(bmi)
                            FROM insurance_data
                            WHERE diabetic = 'Yes'
                              AND children >= 1
                              AND region = 'southwest'));

-- problem 10
SELECT AVG(A.claim - B.claim)
FROM insurance_data A
JOIN insurance_data B
ON A.bmi = B.bmi AND
   A.smoker != B.smoker AND
   A.children = B.children;